# 4 Your own uncertainty, your own decision

The three notebooks before this one each took uncertainty one step further: describe it
(01), guarantee it (02), act on it (03). This one is where you run the whole chain on a
problem of your own and defend the result.

**Pick one of two tracks.** Both are discussed the same way.

| track | data | the decision at the end |
|---|---|---|
| **A — forecasting** | the series you chose in `2_kernels_state_and_gates/04_exercise` | how much to order / reserve, given a predicted distribution |
| **B — detection** | any published cache, as a one-vs-rest task | whether to flag an item, and when to ask a human |

Track B is worked below so you have something that runs. Track A reuses `01_probabilistic`
for the distribution and swaps the final step for a newsvendor decision. If you already did
`2_kernels_state_and_gates/04_exercise` and concluded that your series *needed* a distribution, take
track A — that notebook's last section is the entry point to this one.

In [ ]:
import numpy as np
import torch
from notebooktester import param
from torch import nn
from vectormesh import VectorCache
from vectormesh.components import NeuralNet

# Track B skeleton. Change TARGET (and the cache) to make this your own problem --
# a different class is a genuinely different task, with a different base rate.
CACHE = "pttrn-io/dtd-dinov2-small"
TARGET = 12
BATCH, EPOCHS = 128, param(20, test=1)

cache = VectorCache.from_hub(CACHE)
d = cache.dataset.shuffle(seed=0).with_format("numpy")
X = torch.tensor(np.asarray(d["embed"], dtype="float32"))
y = torch.tensor((np.asarray(d["label"]) == TARGET).astype("int64"))

a, b = int(0.50 * len(X)), int(0.75 * len(X))
Xtr, ytr, Xca, yca, Xte, yte = X[:a], y[:a], X[a:b], y[a:b], X[b:], y[b:]
print(
    f"base rate {float(y.float().mean()):.2%}   "
    f"train {len(Xtr)}  calibration {len(Xca)}  test {len(Xte)}"
)

**Three splits, not two.** Both a conformal threshold and a decision threshold are fitted
quantities, and neither may be fitted on the data you report. This is the single most
common way an uncertainty result turns out to be fiction.

In [ ]:
torch.manual_seed(0)
model = NeuralNet(X.shape[1], 2)
opt = torch.optim.Adam(model.parameters(), lr=1e-3)
loss_fn = nn.CrossEntropyLoss()
for _ in range(EPOCHS):
    model.train()
    perm = torch.randperm(len(Xtr))
    for i in range(0, len(Xtr) - 1, BATCH):
        idx = perm[i : i + BATCH]
        opt.zero_grad()
        loss_fn(model(Xtr[idx]), ytr[idx]).backward()
        opt.step()
model.eval()

with torch.no_grad():
    p_ca = torch.softmax(model(Xca), -1)[:, 1].numpy()
    p_te = torch.softmax(model(Xte), -1)[:, 1].numpy()
yc, yt = yca.numpy(), yte.numpy()

print(f"accuracy at 0.5      : {((p_te >= 0.5).astype(int) == yt).mean():.4f}")
print(f"accuracy saying 'no' : {(yt == 0).mean():.4f}   <- the number to beat")

That is the whole starting point. Everything below is yours.

## 4.1 Is the uncertainty real, or is the model just unsure?

Before any machinery: separate **aleatoric** uncertainty (the world is genuinely variable —
two identical-looking textures really do have different labels) from **epistemic**
uncertainty (the model has not seen enough).

They are not the same and they have different fixes. More data reduces the second and does
nothing to the first. Make the case with one plot before you build anything,
because if your uncertainty is entirely epistemic then the honest recommendation is
"collect more data", and no amount of conformal machinery changes that.

*Track A:* the neighbour-fan plot from `2_kernels_state_and_gates/04_exercise` §4.3 answers this
directly. *Track B:* find the nearest neighbours of a few test items in embedding space and
check whether their labels agree.

## 4.2 Produce a distribution, and score it

Not a point prediction. Depending on your track:

- **A:** a Gaussian head and a quantile head from `01_probabilistic`, scored with CRPS
  against the free baseline that notebook establishes. Reporting a CRPS with no baseline
  is reporting nothing.
- **B:** the class probabilities you already have — but check whether they *mean* anything.
  Bin the test items by predicted probability and compare with the observed frequency in
  each bin. That plot is the calibration check, and on an imbalanced task it is usually
  unflattering.

## 4.3 Guarantee it

Apply split conformal (`02_conformal`) to whatever you built. Measure:

1. **empirical coverage** against the level you asked for, on the test split;
2. **the cost of the guarantee** — average set size, or interval width;
3. **one deliberate violation.** Construct a shift where you expect coverage to break,
   predict the direction *before* running it, and see what happened. §2.5 breaks it by
   calibrating on easy classes and deploying on hard ones; a random split, by contrast,
   barely moved coverage. Yours may do either — say which and why.

## 4.4 Decide, with a price on being wrong

State a cost ratio **in one sentence a domain expert would recognise**, and defend it:

> "In this setting, missing one is worth about N false alarms, because …"

Then, from `03_decisions`:

- the threshold that ratio implies, chosen on calibration and scored on test;
- the expected cost of that threshold against the 0.5 default;
- the confusion matrix at both, so the trade is visible rather than asserted;
- **the break-even review price**, if abstention makes sense for your problem.

*Track A:* the same logic as a quantity rather than a threshold — if unsold stock costs
`C_o` and a stockout costs `C_u`, order the `C_u / (C_u + C_o)` quantile of your predicted
distribution and compare with ordering the mean.

## What you bring

Your runs, discussed in the lesson in mlflow. Between them they cover:

1. Your task, its base rate, and the trivial baseline it has to beat.
2. Evidence about whether the uncertainty is aleatoric or epistemic, with the plot.
3. A scored distribution, against a baseline.
4. A coverage table, including your deliberate violation.
5. Your cost ratio, in words, and the decision it implies with its expected cost.
6. **At least one negative result** you produced and stand behind.

For item 6, the useful shape here is often "I built the machinery and it was not worth
it" — a series whose spread is narrow enough that a point forecast is fine, or a cost ratio
close enough to 1 that the default threshold was already near-optimal. That is a real
finding about your problem, and it is more useful than a conformal interval nobody needed.

## Exercises

**E1 — the calibration split is not free.** Halve it and re-run §4.3. How much does the
achieved coverage wobble across different random calibration subsets of that size? Report
the spread, not one number.

**E2 — where does your ratio stop mattering?** Sweep `C_fn / C_fp` over a wide range and
plot the optimal threshold against it. Over what range of ratios does the decision stay the
same? If your domain's plausible range sits inside a flat region, say so — it means the
exact number does not need defending, which is worth knowing.

**E3 — one seed is an anecdote.** Everything above ran on one seed. Re-run the pipeline
end to end with three, and check whether your headline conclusion survives.

**E4 — the honest alternative.** Take the simplest thing that could have worked — the 0.5
threshold, or the mean forecast — and compute what it would have cost. If the gap is
small, that comparison is the run to lead with.